# 02b - HGT Full Preview (Kaggle)

Run this after `02a_train_hgt_smoke_test.ipynb` passes.

Purpose:
- Use the same graph artifact / mmap CSR training store path.
- Train the proposed IDS-HGT with the Kaggle preview budget.
- Use DDP across both visible Kaggle GPUs when available.

Output bundle: `/kaggle/working/hgt_results.zip` with runtime configs, logs,
checkpoints, summaries, and comparison CSV.


In [ ]:
from pathlib import Path
import torch

# Repo
GITHUB_REPO_URL       = "https://github.com/LeThanhPhat-ATTT2023/Do-an-chuyen-nganh_NT114.git"
GITHUB_BRANCH         = ""
FORCE_RECLONE         = False
GITHUB_PULL_IF_CLONED = True

WORK_DIR     = Path("/kaggle/working/nt114_hgt")
RESULT_ZIP   = Path("/kaggle/working/hgt_results.zip")

# Input filenames. Primary names match the 14GB graph export; fallbacks keep
# older generic artifact names usable.
GRAPH_NPZ_CANDIDATE_NAMES = [
    "graph_artifact_3tier_14gb.npz",
    "graph_artifact_3tier.npz",
]
GRAPH_META_CANDIDATE_NAMES = [
    "graph_artifact_3tier_14gb.meta.json",
    "graph_artifact_3tier.meta.json",
]
GRAPH_PKT_SEM_CANDIDATE_NAMES = [
    "graph_artifact_3tier_14gb_packet_semantic_x.npy.zst",
    "graph_artifact_3tier_14gb_packet_semantic_x.npy",
    "graph_artifact_3tier_packet_semantic_x.npy.zst",
    "graph_artifact_3tier_packet_semantic_x.npy",
]
GRAPH_NPZ_FALLBACK_PATTERNS = ("*graph_artifact_3tier*.npz",)
GRAPH_META_FALLBACK_PATTERNS = ("*graph_artifact_3tier*.meta.json",)
GRAPH_PKT_SEM_FALLBACK_PATTERNS = ("*packet_semantic_x.npy.zst", "*packet_semantic_x.npy", "*packet_semantic_x.npy.gz")

# Decompressed packet features go to /tmp (not counted against /kaggle/working quota).
GRAPH_PKT_SEM_DECOMPRESS_DIR = Path("/tmp")
# Graph store also goes to /tmp; packet features can be much larger than Kaggle working quota.
TRAIN_GRAPH_STORE_ROOT  = Path("/tmp/graph_store_mmap_csr_v1")

# Run profile
SMOKE_TEST            = False
SMOKE_EPOCHS          = 3
SMOKE_MAX_TRAIN_FLOWS = 60000
FULL_PREVIEW_EPOCHS   = 12
FULL_PREVIEW_PATIENCE = 5

# HGT run mode
# "deployment" = one proposed-model config.
# "paper_variants" = multiple comparison configs, one GPU each when possible.
HGT_RUN_MODE = "deployment"

# Memory / OOM protection
SAFE_15GB_PROFILE    = True
MAX_BATCH_SEED_FLOWS = 256
MIN_BATCH_SEED_FLOWS = 16
OOM_RETRIES          = 3
ALLOW_CPU_FALLBACK   = False

# Install flags
INSTALL_WITH_DEPS    = False
INSTALL_MISSING_DEPS = True
USE_INPUT_TRAIN_GRAPH_STORE_IF_PRESENT = True
RESET_OUTPUTS        = False

# GPU
GPU_IDS: list[int] = list(range(torch.cuda.device_count())) or [0]
if SMOKE_TEST:
    GPU_IDS = GPU_IDS[:1]
MAX_PARALLEL_HGT_RUNS: int = len(GPU_IDS)

# Configs. Kaggle config is preferred when present; fallback keeps GitHub clone usable.
KAGGLE_CONFIG   = "configs/hgt_t082_k5_l3_d01_kaggle.yaml"
FALLBACK_CONFIG = "configs/hgt_t082_k5_l3_d01.yaml"

DEPLOYMENT_RUNS = [
    {"name": "nt114_proposed_l4_h256_h8", "config": KAGGLE_CONFIG, "fallback_config": FALLBACK_CONFIG},
]

PAPER_VARIANT_RUNS = [
    {"name": "baseline_t082_k5_l3_d01",          "config": FALLBACK_CONFIG},
    {"name": "xgnid_dual_modal_l1_h32_h4",        "config": "configs/hgt_paper_variants/hgt_t082_k5_xgnid_dual_modal_l1_h32_h4.yaml"},
    {"name": "one2_iov_l1_h64_h2",               "config": "configs/hgt_paper_variants/hgt_t082_k5_one2_iov_l1_h64_h2.yaml"},
    {"name": "relgt_multi_token_l3_h128_h8",     "config": "configs/hgt_paper_variants/hgt_t082_k5_relgt_multi_token_l3_h128_h8.yaml"},
    {"name": "gatransformer_deep_l6_h256_h8",    "config": "configs/hgt_paper_variants/hgt_t082_k5_gatransformer_deep_l6_h256_h8.yaml"},
    {"name": "ahgt_dfd_funnel_l3_h128_h4",       "config": "configs/hgt_paper_variants/hgt_t082_k5_ahgt_dfd_funnel_l3_h128_h4.yaml"},
    {"name": "dlg_ids_sparse_l2_h128_h4",        "config": "configs/hgt_paper_variants/hgt_t082_k5_dlg_ids_sparse_l2_h128_h4.yaml"},
]

RUNS = DEPLOYMENT_RUNS if HGT_RUN_MODE == "deployment" else PAPER_VARIANT_RUNS


In [ ]:
import sys, torch

print("Python:", sys.version)
print("Torch:",  torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  VRAM={p.total_memory/1024**3:.1f} GB")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA not available.\n"
        "Kaggle: Session options → Accelerator → GPU T4 x2"
    )

MAX_PARALLEL_HGT_RUNS = min(MAX_PARALLEL_HGT_RUNS, len(GPU_IDS), len(RUNS))
print(f"HGT_RUN_MODE       : {HGT_RUN_MODE}")
print(f"Active GPU_IDS     : {GPU_IDS}")
print(f"Max parallel runs  : {MAX_PARALLEL_HGT_RUNS}")

In [ ]:
# ── Repo setup + install ───────────────────────────────────────────────────────
import os, shutil, subprocess, sys
from pathlib import Path


def run_streaming(cmd, *, cwd=None, env=None):
    """Run a subprocess and stream output line-by-line; progress bars update in-place."""
    print("\n$", " ".join(str(x) for x in cmd))
    proc = subprocess.Popen(
        [str(x) for x in cmd], cwd=cwd or WORK_DIR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        encoding='utf-8', errors='replace', bufsize=1,
    )
    _prev_progress = False
    for line in proc.stdout:
        text = line.rstrip('\n\r')
        is_progress = '%|' in text
        if is_progress:
            sys.stdout.write('\r' + text if _prev_progress else text)
        else:
            if _prev_progress:
                sys.stdout.write('\n')
            sys.stdout.write(text + '\n')
        sys.stdout.flush()
        _prev_progress = is_progress
    if _prev_progress:
        sys.stdout.write('\n')
    proc.wait()
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)


run_cmd = run_streaming


def is_repo_root(path):
    return (
        (path / "src" / "graphslm_ids").exists()
        and (path / "configs" / "hgt_t082_k5_l3_d01.yaml").exists()
        and (path / "pyproject.toml").exists()
    )


def find_repo_in_kaggle_input():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for root in sorted(input_root.glob("*")):
        if is_repo_root(root):
            return root
        for cfg in root.rglob("configs/hgt_t082_k5_l3_d01.yaml"):
            cand = cfg.parents[1]
            if is_repo_root(cand):
                return cand
    return None


def _prepare_repo_impl():
    _skip_kaggle_input = False

    if FORCE_RECLONE and WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)

    if is_repo_root(WORK_DIR):
        if (WORK_DIR / ".git").exists():
            if GITHUB_PULL_IF_CLONED:
                try:
                    run_streaming(["git", "pull", "--ff-only"], cwd=WORK_DIR)
                except subprocess.CalledProcessError as e:
                    print(f"[warn] git pull failed ({e}), continuing.")
            return
        if GITHUB_PULL_IF_CLONED:
            print("[info] No .git found (copied from Kaggle input); recloning from GitHub.")
            shutil.rmtree(WORK_DIR)
            _skip_kaggle_input = True
        else:
            return

    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)

    if not _skip_kaggle_input:
        source = find_repo_in_kaggle_input()
        if source:
            print("Copy repo:", source, "->", WORK_DIR)
            shutil.copytree(
                source, WORK_DIR,
                ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", ".pytest_cache")
            )
            return

    cmd = ["git", "clone"]
    if GITHUB_BRANCH.strip():
        cmd += ["--branch", GITHUB_BRANCH]
    cmd += [GITHUB_REPO_URL, str(WORK_DIR)]
    run_streaming(cmd, cwd=WORK_DIR.parent)


def prepare_repo():
    import fcntl
    lock_path = WORK_DIR.parent / ".nt114_clone.lock"
    lock_path.parent.mkdir(parents=True, exist_ok=True)
    with open(lock_path, "w") as _lf:
        fcntl.flock(_lf, fcntl.LOCK_EX)
        try:
            _prepare_repo_impl()
        finally:
            fcntl.flock(_lf, fcntl.LOCK_UN)


def import_ok(name):
    try:
        __import__(name)
        return True
    except Exception:
        return False


def install_repo():
    try:
        import zstandard  # noqa: F401
    except ImportError:
        run_streaming([sys.executable, "-m", "pip", "install", "--quiet", "zstandard"])

    missing = [n for n in ["numpy", "pandas", "yaml", "torch", "tqdm"] if not import_ok(n)]
    if missing:
        if not INSTALL_MISSING_DEPS:
            raise RuntimeError(f"Missing deps: {missing}")
        run_streaming([sys.executable, "-m", "pip", "install", "-r", "requirements-ml.txt"])
    cmd = [sys.executable, "-m", "pip", "install", "-e", "."]
    if not INSTALL_WITH_DEPS:
        cmd.append("--no-deps")
    run_streaming(cmd)


prepare_repo()
os.chdir(WORK_DIR)
if RESET_OUTPUTS and (WORK_DIR / "outputs").exists():
    shutil.rmtree(WORK_DIR / "outputs")
install_repo()
print("CWD:", Path.cwd())

In [ ]:
# Locate graph artifact files
import json
from pathlib import Path

PROCESSED_DIR = WORK_DIR / "data" / "processed"
RUNTIME_CONFIG_DIR = WORK_DIR / "outputs" / "kaggle_hgt_configs"
LOG_DIR = WORK_DIR / "outputs" / "hgt_logs"
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


def _unique(paths):
    seen = set()
    out = []
    for p in paths:
        p = Path(p)
        key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def find_input_file(names, fallback_patterns=()):
    if isinstance(names, str):
        names = [names]
    candidates = []
    for name in names:
        candidates.append(PROCESSED_DIR / name)

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for name in names:
            candidates.extend(sorted(input_root.rglob(name)))
        for pattern in fallback_patterns:
            candidates.extend(sorted(input_root.rglob(pattern)))

    for p in _unique(candidates):
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Missing one of {list(names)!r}. Upload the graph files as a Kaggle Dataset."
    )


def find_optional_input_file(names, fallback_patterns=()):
    try:
        return find_input_file(names, fallback_patterns)
    except FileNotFoundError:
        return None


GRAPH_NPZ  = find_input_file(GRAPH_NPZ_CANDIDATE_NAMES, GRAPH_NPZ_FALLBACK_PATTERNS)
GRAPH_META = find_input_file(GRAPH_META_CANDIDATE_NAMES, GRAPH_META_FALLBACK_PATTERNS)
print("GRAPH_NPZ :", GRAPH_NPZ)
print("GRAPH_META:", GRAPH_META)


def find_packet_semantic_npy():
    try:
        meta = json.loads(GRAPH_META.read_text(encoding="utf-8"))
        mp = meta.get("packet_semantic_x_npy")
        if mp and Path(mp).exists():
            return Path(mp)
    except Exception:
        pass
    return find_optional_input_file(GRAPH_PKT_SEM_CANDIDATE_NAMES, GRAPH_PKT_SEM_FALLBACK_PATTERNS)


def decompress_if_needed(found_path):
    name = found_path.name
    out_dir = GRAPH_PKT_SEM_DECOMPRESS_DIR
    out_dir.mkdir(parents=True, exist_ok=True)
    if name.endswith(".npy.zst"):
        import zstandard as zstd
        out_path = out_dir / name[:-4]
        if not out_path.exists():
            print(f"Decompressing zstd {found_path.name} -> {out_path}")
            dctx = zstd.ZstdDecompressor()
            try:
                with found_path.open("rb") as fi, out_path.open("wb") as fo:
                    dctx.copy_stream(fi, fo)
            except BaseException:
                out_path.unlink(missing_ok=True)
                raise
        return out_path
    if name.endswith(".npy.gz"):
        import gzip, shutil as _sh
        out_path = out_dir / name[:-3]
        if not out_path.exists():
            try:
                with gzip.open(found_path, "rb") as fi, out_path.open("wb") as fo:
                    _sh.copyfileobj(fi, fo)
            except BaseException:
                out_path.unlink(missing_ok=True)
                raise
        return out_path
    return found_path


_raw_pkt_sem = find_packet_semantic_npy()
if _raw_pkt_sem:
    GRAPH_PKT_SEM_NPY = decompress_if_needed(_raw_pkt_sem)
    print("GRAPH_PKT_SEM_NPY:", GRAPH_PKT_SEM_NPY, "(source:", _raw_pkt_sem.name, ")")
else:
    GRAPH_PKT_SEM_NPY = None
    print("[warn] packet_semantic_x.npy not found; conversion may fail.")


In [ ]:
# ── Convert NPZ → mmap/CSR graph store ────────────────────────────────────────
import json
from pathlib import Path


def graph_store_layout(root):
    manifest = root / "manifest.json"
    if not manifest.exists():
        return None
    return str(json.loads(manifest.read_text(encoding="utf-8")).get("layout", ""))


def is_training_graph_store(root):
    return graph_store_layout(root) == "numpy_memmap_csr"


def find_input_training_graph_store():
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        return None
    for manifest in sorted(input_root.rglob("manifest.json")):
        root = manifest.parent
        if is_training_graph_store(root):
            return root
    return None


def build_training_graph_store():
    if is_training_graph_store(TRAIN_GRAPH_STORE_ROOT):
        print("Use existing mmap/CSR store:", TRAIN_GRAPH_STORE_ROOT)
        return TRAIN_GRAPH_STORE_ROOT

    if USE_INPUT_TRAIN_GRAPH_STORE_IF_PRESENT:
        input_store = find_input_training_graph_store()
        if input_store:
            print("Use input mmap/CSR store:", input_store)
            return input_store

    cmd = [
        sys.executable, "-u", "-m",
        "graphslm_ids.offline.training.on_disk_graph_store",
        "--graph-npz",       str(GRAPH_NPZ),
        "--graph-meta-json", str(GRAPH_META),
        "--output-root",     str(TRAIN_GRAPH_STORE_ROOT),
    ]
    if GRAPH_PKT_SEM_NPY is not None:
        cmd += ["--packet-semantic-npy", str(GRAPH_PKT_SEM_NPY)]
        cmd += ["--symlink-packet-features"]
    run_cmd(cmd)

    if not is_training_graph_store(TRAIN_GRAPH_STORE_ROOT):
        raise RuntimeError("Converted store must have layout=numpy_memmap_csr")
    return TRAIN_GRAPH_STORE_ROOT


GRAPH_STORE_ROOT = build_training_graph_store()
manifest = json.loads((GRAPH_STORE_ROOT / "manifest.json").read_text(encoding="utf-8"))
print("GRAPH_STORE_ROOT  :", GRAPH_STORE_ROOT)
print("NODE_COUNTS:", manifest.get("node_counts"))
print("EDGE_COUNTS:", manifest.get("edge_counts"))

In [ ]:
# Patch HGT YAML configs for Kaggle environment
import math, os
import yaml
from pathlib import Path


def load_yaml(path):
    return yaml.safe_load(Path(path).read_text(encoding="utf-8")) or {}


def dump_yaml(path, data):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(
        yaml.safe_dump(data, sort_keys=False, allow_unicode=False), encoding="utf-8"
    )


def config_source_path(run_cfg):
    path = WORK_DIR / run_cfg["config"]
    if path.exists():
        return path
    fallback = run_cfg.get("fallback_config")
    if fallback:
        fb = WORK_DIR / fallback
        if fb.exists():
            print(f"[warn] Missing {path}; fallback to {fb}")
            return fb
    raise FileNotFoundError(f"Missing HGT config: {path}")


def patch_hgt_config(run_cfg, retry=0):
    cfg = load_yaml(config_source_path(run_cfg))

    data = cfg.setdefault("data", {})
    data.update(
        source="graph_store",
        graph_npz=str(GRAPH_NPZ),
        graph_meta_json=str(GRAPH_META),
        graph_store_root=str(GRAPH_STORE_ROOT),
        read_sealed_only=True,
        packet_feature=data.get("packet_feature", "semantic"),
        add_reverse_edges=True,
        standardize_flow_features=True,
        use_semantic_edge_weights=True,
    )

    train = cfg.setdefault("train", {})
    train.update(batch_mode="neighbor_sampling", device="cuda", amp=True, activation_checkpointing=True)
    train["amp_dtype"] = train.get("amp_dtype", "auto")
    train["tf32"] = bool(train.get("tf32", True))
    # Keep compile off for Kaggle/DDP variable-shape neighbor batches until a run proves it safe.
    train["compile"] = False
    train["multi_gpu"] = True
    train["ddp_bucket_cap_mb"] = int(train.get("ddp_bucket_cap_mb", 25))
    train["monitor"] = train.get("monitor", "val_macro_f1")
    train["log_every"] = int(train.get("log_every", 1))
    train["output_dir"] = "outputs/nt114_proposed_l4_h256_h8_kaggle_preview"

    if SMOKE_TEST:
        train["epochs"] = SMOKE_EPOCHS
        train["patience"] = SMOKE_EPOCHS + 1
        train["max_train_flows"] = SMOKE_MAX_TRAIN_FLOWS
        train["multi_gpu"] = False
        train["output_dir"] = "outputs/nt114_proposed_l4_h256_h8_smoke_3epochs"
    else:
        train["epochs"] = FULL_PREVIEW_EPOCHS
        train["patience"] = FULL_PREVIEW_PATIENCE
        train.pop("max_train_flows", None)

    base_batch = int(train.get("batch_seed_flows", 128))
    base_grad = int(train.get("grad_accum_steps", 1))
    cap = min(base_batch, MAX_BATCH_SEED_FLOWS) if SAFE_15GB_PROFILE else base_batch
    batch = max(MIN_BATCH_SEED_FLOWS, cap // (2 ** retry))
    train["batch_seed_flows"] = int(batch)
    train["grad_accum_steps"] = max(base_grad, math.ceil((base_batch * base_grad) / batch))

    sampler = cfg.setdefault("sampler", {})
    sampler.setdefault("hops", 4)
    sampler.setdefault("fanouts", {
        "flow__contains__packet": 8,
        "packet__next_packet__packet": 4,
        "packet__matches_technique__technique": 5,
        "flow__matches_technique__technique": 5,
        "technique__belongs_to_tactic__tactic": 1,
    })
    sampler.setdefault("reverse_fanouts", {
        "rev_contains": 3, "rev_next_packet": 2,
        "rev_matches_technique": 2, "rev_belongs_to_tactic": 0,
    })
    sampler.setdefault("always_include_all_tactics", True)
    sampler.setdefault("always_include_all_techniques", True)

    n_cpu_per_run = max(1, (os.cpu_count() or 2) // max(1, MAX_PARALLEL_HGT_RUNS))
    dataloader = cfg.setdefault("dataloader", {})
    user_nw = int(dataloader.get("num_workers", -1))
    nw = user_nw if user_nw < 0 else min(user_nw, n_cpu_per_run)
    dataloader.update(
        num_workers=nw,
        prefetch_factor=int(dataloader.get("prefetch_factor", 2)),
        pin_memory=True,
        persistent_workers=bool(nw != 0),
    )

    suffix = "" if retry == 0 else f"_retry{retry}"
    out_path = RUNTIME_CONFIG_DIR / f"{run_cfg['name']}{suffix}.yaml"
    dump_yaml(out_path, cfg)
    return out_path, cfg


def summary_path(config_path):
    cfg = load_yaml(config_path)
    return WORK_DIR / cfg["train"]["output_dir"] / "training_summary.json"


for run_cfg in RUNS:
    cfg_path, cfg = patch_hgt_config(run_cfg, retry=0)
    t = cfg["train"]
    print(run_cfg["name"],
          "| epochs=", t["epochs"],
          "| output_dir=", t["output_dir"],
          "| batch_seed_flows=", t["batch_seed_flows"],
          "| grad_accum_steps=", t["grad_accum_steps"],
          "| max_train_flows=", t.get("max_train_flows", "all"))


In [ ]:
# ── Train HGT ─────────────────────────────────────────────────────────────────
import os, queue, subprocess, sys, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

OOM_MARKERS = (
    "CUDA out of memory", "torch.OutOfMemoryError",
    "CUBLAS_STATUS_ALLOC_FAILED", "CUDA error: out of memory",
)

_STDOUT_LOCK = threading.Lock()


def log_has_oom(path):
    return path.exists() and any(
        m in path.read_text(encoding="utf-8", errors="ignore") for m in OOM_MARKERS
    )


def _stream_proc(proc, log, prefix=""):
    """Read proc.stdout line-by-line; epoch/tqdm progress lines update in-place; write to log."""
    _prev_progress = False
    for line in proc.stdout:
        text = line.rstrip('\n\r')
        # tqdm bars contain '%|'; epoch progress lines contain '%) |' (e.g. "( 10.0%) | loss=")
        is_progress = '%|' in text or ('Epoch ' in text and '%) |' in text)
        display = f"{prefix}{text}" if prefix else text
        with _STDOUT_LOCK:
            if is_progress:
                sys.stdout.write(('\r' + display) if _prev_progress else display)
            else:
                if _prev_progress:
                    sys.stdout.write('\n')
                sys.stdout.write(display + '\n')
            sys.stdout.flush()
        log.write(line)
        log.flush()
        _prev_progress = is_progress
    with _STDOUT_LOCK:
        if _prev_progress:
            sys.stdout.write('\n')
            sys.stdout.flush()


def train_process(run_cfg, config_path, gpu_id, retry):
    log_path = LOG_DIR / f"{run_cfg['name']}_gpu{gpu_id}_retry{retry}.log"
    n_threads = max(1, (os.cpu_count() or 2) // max(1, MAX_PARALLEL_HGT_RUNS))
    env = os.environ.copy()
    env.update(
        PYTHONUNBUFFERED="1",
        PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True,max_split_size_mb:128",
        CUDA_VISIBLE_DEVICES=str(gpu_id),
        OMP_NUM_THREADS=str(n_threads),
        MKL_NUM_THREADS=str(n_threads),
    )
    cmd = [
        sys.executable, "-u", "-m",
        "graphslm_ids.offline.training.train_hgt_flow_classifier",
        "--config", str(config_path),
        "--device", "cuda",
    ]
    print(f"\n=== TRAIN {run_cfg['name']} gpu={gpu_id} retry={retry} ===")
    print("log:", log_path)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(
            cmd, cwd=WORK_DIR,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            encoding='utf-8', errors='replace', bufsize=1, env=env,
        )
        _stream_proc(proc, log, prefix=f"[gpu{gpu_id}:{run_cfg['name']}] ")
        code = proc.wait()
    return code, log_path


def train_ddp(run_cfg, gpu_ids, retry):
    cfg_path, _ = patch_hgt_config(run_cfg, retry=retry)
    log_path = LOG_DIR / f"{run_cfg['name']}_ddp_retry{retry}.log"
    n_threads = max(1, (os.cpu_count() or 2) // max(1, len(gpu_ids)))
    env = os.environ.copy()
    env.update(
        PYTHONUNBUFFERED="1",
        PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True,max_split_size_mb:128",
        CUDA_VISIBLE_DEVICES=",".join(str(g) for g in gpu_ids),
        OMP_NUM_THREADS=str(n_threads),
        MKL_NUM_THREADS=str(n_threads),
    )
    cmd = [
        "torchrun", "--standalone", f"--nproc_per_node={len(gpu_ids)}",
        "-m", "graphslm_ids.offline.training.train_hgt_flow_classifier",
        "--config", str(cfg_path),
    ]
    print(f"\n=== DDP TRAIN {run_cfg['name']} GPUs={gpu_ids} retry={retry} ===")
    print("log:", log_path)
    with log_path.open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(
            cmd, cwd=WORK_DIR,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            encoding='utf-8', errors='replace', bufsize=1, env=env,
        )
        _stream_proc(proc, log, prefix=f"[ddp:{run_cfg['name']}] ")
        return proc.wait(), log_path


def train_one(run_cfg, gpu_id):
    first_cfg, _ = patch_hgt_config(run_cfg, retry=0)
    if summary_path(first_cfg).exists():
        return {"run": run_cfg["name"], "status": "skipped", "summary": str(summary_path(first_cfg))}
    last_log = None
    for retry in range(OOM_RETRIES + 1):
        cfg_path, _ = patch_hgt_config(run_cfg, retry=retry)
        code, log_path = train_process(run_cfg, cfg_path, gpu_id, retry)
        last_log = log_path
        if code == 0:
            return {"run": run_cfg["name"], "status": "ok", "summary": str(summary_path(cfg_path))}
        if not log_has_oom(log_path) or retry == OOM_RETRIES:
            break
        print("OOM detected; retrying with smaller batch_seed_flows.")
    raise RuntimeError(f"HGT run failed: {run_cfg['name']}. See {last_log}")


def train_all():
    if len(RUNS) == 1 and len(GPU_IDS) > 1:
        run_cfg = RUNS[0]
        first_cfg, _ = patch_hgt_config(run_cfg, retry=0)
        if summary_path(first_cfg).exists():
            return [{"run": run_cfg["name"], "status": "skipped",
                     "summary": str(summary_path(first_cfg))}]
        last_log = None
        for retry in range(OOM_RETRIES + 1):
            code, log_path = train_ddp(run_cfg, GPU_IDS, retry)
            last_log = log_path
            if code == 0:
                cfg_path, _ = patch_hgt_config(run_cfg, retry=retry)
                return [{"run": run_cfg["name"], "status": "ok_ddp",
                         "summary": str(summary_path(cfg_path))}]
            if not log_has_oom(log_path) or retry == OOM_RETRIES:
                break
            print("OOM detected; retrying.")
        raise RuntimeError(f"HGT DDP run failed: {run_cfg['name']}. See {last_log}")

    gpu_queue: queue.Queue = queue.Queue()
    for gpu_id in GPU_IDS[:MAX_PARALLEL_HGT_RUNS]:
        gpu_queue.put(gpu_id)

    results = []

    def worker(run_cfg):
        gpu_id = gpu_queue.get()
        try:
            return train_one(run_cfg, gpu_id)
        finally:
            gpu_queue.put(gpu_id)

    with ThreadPoolExecutor(max_workers=MAX_PARALLEL_HGT_RUNS) as ex:
        futures = [ex.submit(worker, r) for r in RUNS]
        for f in as_completed(futures):
            result = f.result()
            results.append(result)
            print("DONE:", result)
    return results


train_results = train_all()
train_results

In [ ]:
# Collect metrics + bundle results
import csv, json, zipfile
from pathlib import Path


def build_comparison():
    rows = []
    for run_cfg in RUNS:
        cfg_path, _ = patch_hgt_config(run_cfg, retry=0)
        sp = summary_path(cfg_path)
        if not sp.exists():
            continue
        data = json.loads(sp.read_text(encoding="utf-8"))
        cfg = data["config"]
        model = cfg["model"]
        t = cfg["train"]
        rows.append({
            "run_name": run_cfg["name"],
            "run_dir": str(sp.parent.relative_to(WORK_DIR)),
            "hidden_dim": model["hidden_dim"],
            "num_layers": model["num_layers"],
            "num_heads": model["num_heads"],
            "epochs": t.get("epochs"),
            "max_train_flows": t.get("max_train_flows", "all"),
            "batch_mode": t.get("batch_mode"),
            "batch_seed_flows": t.get("batch_seed_flows"),
            "grad_accum_steps": t.get("grad_accum_steps"),
            "best_epoch": data.get("best_epoch"),
            "val_macro_f1": data.get("best_val_metrics", {}).get("macro_f1"),
            "test_macro_f1": data.get("best_test_metrics", {}).get("macro_f1"),
            "test_accuracy": data.get("best_test_metrics", {}).get("accuracy"),
            "device": data.get("device"),
        })
    out_dir = WORK_DIR / "outputs"
    out_dir.mkdir(parents=True, exist_ok=True)
    if rows:
        csv_path = out_dir / "hgt_comparison.csv"
        with csv_path.open("w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
        print("Comparison CSV:", csv_path)
    return rows


def bundle_results():
    if RESULT_ZIP.exists():
        RESULT_ZIP.unlink()
    include = [
        WORK_DIR / "outputs",
        GRAPH_META,
        GRAPH_STORE_ROOT / "manifest.json",
        WORK_DIR / KAGGLE_CONFIG,
        WORK_DIR / FALLBACK_CONFIG,
        WORK_DIR / "configs" / "hgt_paper_variants",
    ]
    with zipfile.ZipFile(RESULT_ZIP, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
        for root in include:
            if not root.exists():
                continue
            if root.is_file():
                try:
                    arcname = root.relative_to(WORK_DIR)
                except ValueError:
                    arcname = Path(root.name)
                zf.write(root, arcname)
                continue
            for p in root.rglob("*"):
                if p.is_file():
                    zf.write(p, p.relative_to(WORK_DIR))
    print("Bundle:", RESULT_ZIP)
    print(f"Size: {RESULT_ZIP.stat().st_size / 1024**2:.1f} MB")
    return RESULT_ZIP


comparison_rows = build_comparison()
bundle_results()
comparison_rows


## Notes

- Required Kaggle input: graph NPZ, graph meta JSON, and packet semantic `.npy` or `.npy.zst`.
  The notebook first looks for `graph_artifact_3tier_14gb.*`, then older generic names.
- Graph store is written to `/tmp/graph_store_mmap_csr_v1`, not to runtime `data/graph_store_v1`.
- `02a` smoke mode forces `epochs=3`, `max_train_flows=60000`, and one GPU.
- `02b` full preview mode forces `epochs=12`, removes `max_train_flows`, and uses DDP when two GPUs are visible.
- OOM retry halves `batch_seed_flows` and preserves effective batch through `grad_accum_steps`.
- Re-running skips a run when its `training_summary.json` already exists. Set `RESET_OUTPUTS=True` to retrain.
